In [8]:
from elasticsearch import Elasticsearch

import spacy
import numpy as np

import socket
import requests

In [9]:
def is_elasticsearch_ready():
    try:
        socket.getaddrinfo("elasticsearch", None)
        host = "elasticsearch"
    except socket.gaierror:
        # Fallback to local machine if Docker network host isn't found
        host = "localhost"

    try:
        url = f'http://{host}:9200'
        print(url)

        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Elasticsearch service: {e}")
        return False

is_elasticsearch_ready()

http://localhost:9200


True

In [50]:
embedder = spacy.load('en_core_web_sm')

In [51]:
def get_vector(query):
    nlp = embedder
    doc = nlp(query)
    tokens = [token.lemma_ for token in doc]
    text = ' '.join(tokens)
    doc_lemmatized = nlp(text)
    vector = np.mean([token.vector for token in doc_lemmatized], axis=0).tolist()
    return vector

In [28]:
def _elastic_search_text(query, index_name="documents", num_results=5):
        search_query = {
            "size": num_results,
            "query": {
                "bool": {
                    "must": {
                        "multi_match": {
                            "query": query,
                            "fields": ["question^3", "answer"],
                            "type": "best_fields",
                        }   
                    },
                }
            }
        }

        response = es_client.search(index=index_name, body=search_query)
        return [hit["_source"] for hit in response["hits"]["hits"]]

In [ ]:
def _elastic_search_knn(query,index_name="documents", num_results=5):
        vector = get_vector(query)

        search_body = {
            "knn": {
                "field": "embedding",
                "query_vector": vector,
                "k": num_results,
                "num_candidates": 10000,
            },
            "size": num_results,
            "_source": ['document_id', 'question', 'answer'],
        }

        es_results = es_client.search(index=index_name, body=search_body)

        return [hit["_source"] for hit in es_results["hits"]["hits"]]

In [30]:
def _elastic_search_hybrid(query,index_name="documents", num_results=5):
        vector = get_vector(query)
    
        knn_query = {
            "field": "embedding",
            "query_vector": vector,
            "k": num_results,
            "num_candidates": 10000,
            "boost": 0.5,
        }

        keyword_query = {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "answer"],
                        "type": "best_fields",
                        "boost": 0.5,
                    }   
                },
            }
        }

        es_results = es_client.search(
            index=index_name,
            query=keyword_query,
            knn=knn_query,
            size=num_results
        )

        return [hit["_source"] for hit in es_results["hits"]["hits"]]

In [32]:
def search(search_type, query, num_results=5):
    if search_type == 'text':
        search_results = _elastic_search_text(query)
    elif search_type == 'vector':
        search_results = _elastic_search_knn(query)
    elif search_type == 'hybrid':
        search_results = _elastic_search_hybrid(query)
    
    return search_results

In [55]:
query = "How can i create my account?"
es_client = Elasticsearch('http://localhost:9200') 
index_name='documents'



In [60]:
search_results = search("hybrid",query)
doc = search_results[0]

In [61]:
len(search_results)

5

In [58]:
def build_context(search_results):
        lines = []

        for doc in search_results:
            lines.append('Q: ' + doc['question'])
            lines.append('A: ' + doc['answer'])
            lines.append('')
        
        return '\n'.join(lines).strip()

In [62]:
build_context(search_results)

"Q: How can I create an account?\nA: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.\n\nQ: How can I track my order?\nA: You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.\n\nQ: Can I order without creating an account?\nA: Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.\n\nQ: How can I contact customer support?\nA: You can contact our customer support team by phone at [phone number] or by email at [email address]. Our team is available [working hours] to assist you with any inquiries or issues you may have.\n\nQ: How can I leave a product review?\nA: To leave a product review, navigate to the product page on our website and click on the 'Write a Revie

In [80]:
def get_postgres_host():
    try:
        socket.getaddrinfo("postgres", None)
        host = "postgres"
    except socket.gaierror:
        # Fallback to local machine if Docker network host isn't found
        host = "localhost"
    
    return host  

In [81]:
def get_db_connection():
    dynamic_host = get_postgres_host()
    return psycopg.connect(
        host=dynamic_host,
        dbname=os.getenv("POSTGRES_DB", "ecommerce_chatbot"),
        user=os.getenv("POSTGRES_USER", "user"),
        password=os.getenv("POSTGRES_PASSWORD", "password"),
    )

In [82]:
import psycopg
import os
get_db_connection()

<psycopg.Connection [IDLE] (host=localhost user=user database=ecommerce_chatbot) at 0x7145e1b53ec0>